# 🇹🇭 Thai NLP Inference - Unified Models for FastAPI/Docker

**Notebook นี้รวม 3 Thai NLP Models สำหรับใช้ใน Production Web App**

## Models:
| # | Model | Task | Type |
|---|-------|------|------|
| 1 | `unduood/phayathaibert-absa-sports-facility-v2` | Aspect-Based Sentiment Analysis | Multi-aspect |
| 2 | `unduood/phayathaibert-intent-classification-sports-facility` | Intent Classification | Multi-label |
| 3 | `poom-sci/WangchanBERTa-finetuned-sentiment` | Sentiment Analysis | Single-label |

## เหมาะสำหรับ:
- FastAPI backend services
- Docker containerized applications
- Production-ready inference

---

## 📦 1. Install Dependencies

In [ ]:
# สำหรับ Docker/Production ให้ใส่ใน requirements.txt แทน
# !pip install torch transformers huggingface_hub sentencepiece -q

In [ ]:
import torch
import torch.nn.functional as F
import json
from typing import List, Dict, Union, Optional
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import hf_hub_download
import warnings

warnings.filterwarnings('ignore')

# Auto-detect device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {DEVICE}")

---
## 🎯 2. Model Configuration

กำหนดค่า Model Names ที่จะใช้

In [ ]:
# =====================================================
# MODEL CONFIGURATION - แก้ไขตรงนี้ถ้าต้องการเปลี่ยน model
# =====================================================

MODEL_CONFIGS = {
    "absa": {
        "repo_id": "unduood/phayathaibert-absa-sports-facility-v2",
        "description": "Aspect-Based Sentiment Analysis for Sports Facility"
    },
    "intent": {
        "repo_id": "unduood/phayathaibert-intent-classification-sports-facility",
        "description": "Intent Classification (Feedback/Complaint/Question/Off-topic)"
    },
    "sentiment": {
        "repo_id": "poom-sci/WangchanBERTa-finetuned-sentiment",
        "description": "General Thai Sentiment Analysis"
    }
}

print("📋 Models to load:")
for name, config in MODEL_CONFIGS.items():
    print(f"   • {name}: {config['repo_id']}")

---
## 🔧 3. Service Classes (Production-Ready)

### แต่ละ Class สามารถนำไปใช้ใน FastAPI Service ได้โดยตรง

### 3.1 ABSA Service (Aspect-Based Sentiment Analysis)

In [ ]:
@dataclass
class ABSAResult:
    """Result structure for ABSA prediction"""
    aspect: str
    aspect_thai: str
    sentiment: str
    confidence: float


class ABSAService:
    """
    Aspect-Based Sentiment Analysis Service
    
    วิเคราะห์ความรู้สึกต่อ Aspect ต่างๆ เช่น:
    - Equipment (อุปกรณ์)
    - Staff (พนักงาน)
    - Cleanliness (ความสะอาด)
    - etc.
    
    Usage:
        service = ABSAService()
        result = service.analyze("เครื่องดีมาก พนักงานไม่ค่อยดี")
    """
    
    def __init__(
        self, 
        repo_id: str = MODEL_CONFIGS["absa"]["repo_id"],
        device: Optional[str] = None
    ):
        self.device = torch.device(device or DEVICE)
        self.repo_id = repo_id
        
        # Load tokenizer & model
        self.tokenizer = AutoTokenizer.from_pretrained(repo_id)
        self.model = AutoModelForSequenceClassification.from_pretrained(repo_id)
        self.model.to(self.device)
        self.model.eval()
        
        # Load ABSA config (aspects, labels)
        config_path = hf_hub_download(repo_id=repo_id, filename="absa_config.json")
        with open(config_path, 'r', encoding='utf-8') as f:
            config = json.load(f)
        
        self.aspects = config['aspects']  # {'Equipment': 'อุปกรณ์', ...}
        self.aspect_list = config['aspect_list']
        self.sentiment_labels = config['sentiment_labels']  # ['none', 'positive', 'negative', 'neutral']
        self.max_length = config.get('max_length', 128)
        
        print(f"✅ ABSAService loaded: {repo_id}")
        print(f"   Aspects: {self.aspect_list}")
        print(f"   Labels: {self.sentiment_labels}")
    
    def _predict_single_aspect(self, text: str, aspect_th: str) -> Dict:
        """Internal: predict sentiment for one aspect"""
        encoding = self.tokenizer(
            text, 
            aspect_th,  # sentence pair: [text] [SEP] [aspect]
            return_tensors='pt',
            truncation=True,
            padding='max_length',
            max_length=self.max_length
        )
        
        with torch.no_grad():
            outputs = self.model(
                input_ids=encoding['input_ids'].to(self.device),
                attention_mask=encoding['attention_mask'].to(self.device)
            )
            probs = F.softmax(outputs.logits, dim=-1)[0]
            pred_idx = torch.argmax(probs).item()
        
        return {
            'sentiment': self.sentiment_labels[pred_idx],
            'confidence': probs[pred_idx].item(),
            'probabilities': {label: probs[i].item() for i, label in enumerate(self.sentiment_labels)}
        }
    
    def analyze(self, text: str, include_none: bool = False) -> List[Dict]:
        """
        วิเคราะห์ความรู้สึกต่อทุก Aspect
        
        Args:
            text: ข้อความที่ต้องการวิเคราะห์
            include_none: รวม aspect ที่ไม่พบด้วย (default: False)
        
        Returns:
            List of {aspect, aspect_thai, sentiment, confidence}
        """
        results = []
        
        for aspect_en in self.aspect_list:
            aspect_th = self.aspects[aspect_en]
            pred = self._predict_single_aspect(text, aspect_th)
            
            if include_none or pred['sentiment'] != 'none':
                results.append({
                    'aspect': aspect_en,
                    'aspect_thai': aspect_th,
                    'sentiment': pred['sentiment'],
                    'confidence': round(pred['confidence'], 4)
                })
        
        return results
    
    def analyze_with_probabilities(self, text: str) -> Dict:
        """
        วิเคราะห์พร้อม probability ทั้งหมด (สำหรับ debug/analysis)
        
        Returns:
            {aspect: {sentiment, confidence, probabilities}}
        """
        results = {}
        for aspect_en in self.aspect_list:
            aspect_th = self.aspects[aspect_en]
            results[aspect_en] = self._predict_single_aspect(text, aspect_th)
        return results

### 3.2 Intent Classification Service

In [ ]:
@dataclass
class IntentResult:
    """Result structure for Intent Classification"""
    labels: List[str]
    probabilities: Dict[str, float]


class IntentClassificationService:
    """
    Multi-Label Intent Classification Service
    
    จำแนกประเภทข้อความ:
    - Feedback: ความคิดเห็น/รีวิว
    - Complaint: ข้อร้องเรียนร้ายแรง
    - Question: คำถาม/สอบถาม
    - Off-topic: ไม่เกี่ยวข้อง
    
    ⚠️ Multi-label: ข้อความเดียวอาจมีหลาย intent
    
    Usage:
        service = IntentClassificationService()
        result = service.classify("อุปกรณ์เก่าไปหน่อย มีแพ็คเกจใหม่ไหม")
    """
    
    def __init__(
        self, 
        repo_id: str = MODEL_CONFIGS["intent"]["repo_id"],
        device: Optional[str] = None,
        default_threshold: float = 0.5
    ):
        self.device = torch.device(device or DEVICE)
        self.repo_id = repo_id
        self.default_threshold = default_threshold
        
        # Load tokenizer & model
        self.tokenizer = AutoTokenizer.from_pretrained(repo_id)
        self.model = AutoModelForSequenceClassification.from_pretrained(repo_id)
        self.model.to(self.device)
        self.model.eval()
        
        # Get labels from model config
        self.labels = list(self.model.config.id2label.values())
        self.max_length = 128
        
        print(f"✅ IntentClassificationService loaded: {repo_id}")
        print(f"   Labels: {self.labels}")
        print(f"   Default threshold: {default_threshold}")
    
    def classify(
        self, 
        text: Union[str, List[str]], 
        threshold: Optional[float] = None
    ) -> Union[Dict, List[Dict]]:
        """
        จำแนก intent ของข้อความ
        
        Args:
            text: ข้อความหรือ list ของข้อความ
            threshold: threshold สำหรับ multi-label (default: 0.5)
        
        Returns:
            Dict หรือ List[Dict] ของ {labels, probabilities}
        """
        threshold = threshold or self.default_threshold
        single_input = isinstance(text, str)
        texts = [text] if single_input else text
        
        # Tokenize
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Inference (Multi-label uses sigmoid, not softmax)
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
        
        # Build results
        results = []
        for i, txt in enumerate(texts):
            predicted_labels = []
            probabilities = {}
            
            for j, label in enumerate(self.labels):
                prob = float(probs[i][j])
                probabilities[label] = round(prob, 4)
                if prob >= threshold:
                    predicted_labels.append(label)
            
            results.append({
                'text': txt,
                'labels': predicted_labels,
                'probabilities': probabilities
            })
        
        return results[0] if single_input else results
    
    def get_primary_intent(self, text: str) -> Dict:
        """
        หา intent หลัก (probability สูงสุด)
        
        Returns:
            {label, confidence}
        """
        result = self.classify(text, threshold=0.0)
        probs = result['probabilities']
        primary = max(probs, key=probs.get)
        return {
            'label': primary,
            'confidence': probs[primary]
        }

### 3.3 Sentiment Analysis Service

In [ ]:
@dataclass
class SentimentResult:
    """Result structure for Sentiment Analysis"""
    label: str
    confidence: float
    probabilities: Dict[str, float]


class SentimentAnalysisService:
    """
    Thai Sentiment Analysis Service (WangchanBERTa)
    
    วิเคราะห์ความรู้สึกโดยรวมของข้อความ:
    - pos: เชิงบวก
    - neg: เชิงลบ  
    - neu: กลางๆ
    
    Usage:
        service = SentimentAnalysisService()
        result = service.analyze("อาหารอร่อยมาก")
    """
    
    def __init__(
        self, 
        repo_id: str = MODEL_CONFIGS["sentiment"]["repo_id"],
        device: Optional[str] = None
    ):
        self.device = torch.device(device or DEVICE)
        self.repo_id = repo_id
        
        # Load tokenizer & model
        self.tokenizer = AutoTokenizer.from_pretrained(repo_id)
        self.model = AutoModelForSequenceClassification.from_pretrained(repo_id)
        self.model.to(self.device)
        self.model.eval()
        
        # Get labels from model config
        self.id2label = self.model.config.id2label
        self.labels = list(self.id2label.values())
        self.max_length = 512
        
        print(f"✅ SentimentAnalysisService loaded: {repo_id}")
        print(f"   Labels: {self.labels}")
    
    def analyze(
        self, 
        text: Union[str, List[str]],
        return_all_probs: bool = False
    ) -> Union[Dict, List[Dict]]:
        """
        วิเคราะห์ sentiment ของข้อความ
        
        Args:
            text: ข้อความหรือ list ของข้อความ
            return_all_probs: คืนค่า probability ทุก class
        
        Returns:
            Dict หรือ List[Dict] ของ {label, confidence, [probabilities]}
        """
        single_input = isinstance(text, str)
        texts = [text] if single_input else text
        
        # Tokenize
        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
            padding=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Inference (Single-label uses softmax)
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        
        # Build results
        results = []
        for i, txt in enumerate(texts):
            pred_idx = probs[i].argmax()
            result = {
                'text': txt,
                'label': self.id2label[pred_idx],
                'confidence': round(float(probs[i][pred_idx]), 4)
            }
            
            if return_all_probs:
                result['probabilities'] = {
                    self.id2label[j]: round(float(probs[i][j]), 4)
                    for j in range(len(self.labels))
                }
            
            results.append(result)
        
        return results[0] if single_input else results

---
## 🚀 4. Unified NLP Service (รวมทุก Model)

### Class นี้เหมาะสำหรับใช้เป็น singleton ใน FastAPI

In [ ]:
class ThaiNLPService:
    """
    🇹🇭 Unified Thai NLP Service
    
    รวม 3 models สำหรับวิเคราะห์ข้อความภาษาไทย:
    1. ABSA - วิเคราะห์ความรู้สึกต่อ Aspect ต่างๆ
    2. Intent - จำแนกประเภทข้อความ
    3. Sentiment - วิเคราะห์ความรู้สึกโดยรวม
    
    Usage (FastAPI):
        # ใน main.py หรือ dependencies.py
        nlp_service = ThaiNLPService()
        
        @app.post("/analyze")
        def analyze(text: str):
            return nlp_service.analyze_all(text)
    """
    
    def __init__(self, device: Optional[str] = None, lazy_load: bool = False):
        """
        Initialize Thai NLP Service
        
        Args:
            device: 'cuda' หรือ 'cpu' (auto-detect if None)
            lazy_load: True = โหลด model เมื่อใช้งานครั้งแรก
        """
        self.device = device
        self._absa = None
        self._intent = None
        self._sentiment = None
        
        if not lazy_load:
            self._load_all_models()
    
    def _load_all_models(self):
        """Load all models"""
        print("\n" + "="*60)
        print("🔄 Loading Thai NLP Models...")
        print("="*60 + "\n")
        
        self._absa = ABSAService(device=self.device)
        print()
        self._intent = IntentClassificationService(device=self.device)
        print()
        self._sentiment = SentimentAnalysisService(device=self.device)
        
        print("\n" + "="*60)
        print("✅ All models loaded successfully!")
        print("="*60)
    
    @property
    def absa(self) -> ABSAService:
        """Get ABSA service (lazy load if needed)"""
        if self._absa is None:
            self._absa = ABSAService(device=self.device)
        return self._absa
    
    @property
    def intent(self) -> IntentClassificationService:
        """Get Intent service (lazy load if needed)"""
        if self._intent is None:
            self._intent = IntentClassificationService(device=self.device)
        return self._intent
    
    @property
    def sentiment(self) -> SentimentAnalysisService:
        """Get Sentiment service (lazy load if needed)"""
        if self._sentiment is None:
            self._sentiment = SentimentAnalysisService(device=self.device)
        return self._sentiment
    
    def analyze_all(self, text: str) -> Dict:
        """
        🔮 วิเคราะห์ครบทุก model ในครั้งเดียว
        
        Returns:
            {
                "text": str,
                "sentiment": {label, confidence},
                "intent": {labels, probabilities},
                "aspects": [{aspect, sentiment, confidence}, ...]
            }
        """
        return {
            "text": text,
            "sentiment": {
                "label": self.sentiment.analyze(text)['label'],
                "confidence": self.sentiment.analyze(text)['confidence']
            },
            "intent": self.intent.classify(text),
            "aspects": self.absa.analyze(text)
        }

---
## 🎮 5. Demo: Load & Test All Models

In [ ]:
# Initialize unified service (will load all 3 models)
nlp = ThaiNLPService()

### 5.1 Test ABSA (Aspect-Based Sentiment)

In [ ]:
print("\n" + "="*60)
print("🔮 TEST: ABSA (Aspect-Based Sentiment Analysis)")
print("="*60)

test_texts = [
    "เครื่องดีมาก พนักงานน่ารัก แต่แพงไป",
    "สะอาดดี บรรยากาศดีมาก แต่ที่จอดรถน้อย",
    "คลาสโยคะสนุกมาก ครูสอนดี"
]

for text in test_texts:
    print(f"\n📝 Input: {text}")
    results = nlp.absa.analyze(text)
    print(f"   Found {len(results)} aspects:")
    for r in results:
        emoji = {'positive': '😊', 'negative': '😞', 'neutral': '😐'}.get(r['sentiment'], '')
        print(f"   • {r['aspect']} ({r['aspect_thai']}): {r['sentiment']} {emoji} ({r['confidence']:.1%})")

### 5.2 Test Intent Classification

In [ ]:
print("\n" + "="*60)
print("🎯 TEST: Intent Classification")
print("="*60)

test_texts = [
    "สนามดีมากครับ อุปกรณ์ครบ",          # Feedback
    "แอร์เสียมา 3 วันแล้ว ไม่มีคนมาซ่อม",  # Complaint
    "มีคลาสโยคะวันอาทิตย์ไหมคะ",          # Question
    "555 โอเค",                          # Off-topic
    "อุปกรณ์เก่าไปหน่อย มีแพ็คเกจใหม่ไหม"   # Feedback + Question
]

for text in test_texts:
    result = nlp.intent.classify(text)
    labels_str = ', '.join(result['labels']) if result['labels'] else 'None'
    print(f"\n📝 Input: {text}")
    print(f"   Labels: [{labels_str}]")
    print(f"   Probs: {result['probabilities']}")

### 5.3 Test Sentiment Analysis

In [ ]:
print("\n" + "="*60)
print("💭 TEST: Sentiment Analysis")
print("="*60)

test_texts = [
    "อาหารร้านนี้อร่อยมากๆ เลย บริการดีมาก",
    "ผิดหวังมาก สินค้าไม่ตรงปก",
    "ก็ธรรมดานะ ไม่ได้รู้สึกอะไร",
    "แย่มากเลย ไม่ซื้ออีกแล้ว",
    "ขอบคุณมากครับ ประทับใจมาก"
]

results = nlp.sentiment.analyze(test_texts, return_all_probs=True)

for r in results:
    emoji = {'pos': '😊', 'neg': '😞', 'neu': '😐'}.get(r['label'], '')
    print(f"\n📝 Input: {r['text']}")
    print(f"   Sentiment: {r['label']} {emoji} ({r['confidence']:.1%})")
    print(f"   Probs: {r['probabilities']}")

### 5.4 Test Combined Analysis

In [ ]:
print("\n" + "="*60)
print("🔮 TEST: Combined Analysis (All Models)")
print("="*60)

text = "เครื่องออกกำลังกายดีมาก พนักงานน่ารัก แต่ที่จอดรถน้อยไป มีโปรโมชั่นใหม่ไหมครับ"
print(f"\n📝 Input: {text}")

result = nlp.analyze_all(text)

print(f"\n📊 Results:")
print(f"\n   💭 Overall Sentiment: {result['sentiment']['label']} ({result['sentiment']['confidence']:.1%})")
print(f"\n   🎯 Intent: {result['intent']['labels']}")
print(f"\n   📋 Aspects ({len(result['aspects'])} found):")
for a in result['aspects']:
    print(f"      • {a['aspect']}: {a['sentiment']} ({a['confidence']:.1%})")

---
## 📋 6. FastAPI Integration Example

### ตัวอย่าง Code สำหรับใช้ใน FastAPI

In [ ]:
# ============================================================
# ตัวอย่าง FastAPI Integration
# บันทึกเป็น nlp_service.py ใน project ของคุณ
# ============================================================

FASTAPI_EXAMPLE = '''
# File: app/services/nlp_service.py

from typing import Optional
import torch

# Import service classes (จาก notebook นี้)
# from app.services.thai_nlp import ThaiNLPService

# Singleton instance
_nlp_service: Optional[ThaiNLPService] = None

def get_nlp_service() -> ThaiNLPService:
    """Get or create NLP service singleton"""
    global _nlp_service
    if _nlp_service is None:
        _nlp_service = ThaiNLPService()
    return _nlp_service


# File: app/routers/nlp.py

from fastapi import APIRouter, Depends
from pydantic import BaseModel
from typing import List, Dict

router = APIRouter(prefix="/api/nlp", tags=["NLP"])

class TextInput(BaseModel):
    text: str

class TextsInput(BaseModel):
    texts: List[str]

@router.post("/analyze")
def analyze_text(input: TextInput):
    """วิเคราะห์ข้อความด้วยทุก model"""
    nlp = get_nlp_service()
    return nlp.analyze_all(input.text)

@router.post("/sentiment")
def analyze_sentiment(input: TextInput):
    """วิเคราะห์ sentiment"""
    nlp = get_nlp_service()
    return nlp.sentiment.analyze(input.text)

@router.post("/sentiment/batch")
def analyze_sentiment_batch(input: TextsInput):
    """วิเคราะห์ sentiment หลายข้อความ"""
    nlp = get_nlp_service()
    return nlp.sentiment.analyze(input.texts)

@router.post("/intent")
def classify_intent(input: TextInput, threshold: float = 0.5):
    """จำแนก intent"""
    nlp = get_nlp_service()
    return nlp.intent.classify(input.text, threshold=threshold)

@router.post("/absa")
def analyze_aspects(input: TextInput):
    """วิเคราะห์ ABSA"""
    nlp = get_nlp_service()
    return nlp.absa.analyze(input.text)


# File: app/main.py

from fastapi import FastAPI
from app.routers import nlp
from app.services.nlp_service import get_nlp_service

app = FastAPI(title="Thai NLP API")
app.include_router(nlp.router)

@app.on_event("startup")
async def startup():
    """Pre-load models on startup"""
    get_nlp_service()
'''

print(FASTAPI_EXAMPLE)

---
## 🐳 7. Dockerfile Example

In [ ]:
DOCKERFILE_EXAMPLE = '''
# Dockerfile
FROM python:3.10-slim

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    gcc \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements
COPY requirements.txt .

# Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy app code
COPY . .

# Pre-download models (optional, but recommended)
RUN python -c "from app.services.nlp_service import get_nlp_service; get_nlp_service()"

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

REQUIREMENTS_EXAMPLE = '''
# requirements.txt
fastapi>=0.100.0
uvicorn>=0.22.0
torch>=2.0.0
transformers>=4.30.0
huggingface_hub>=0.16.0
sentencepiece>=0.1.99
pydantic>=2.0.0
'''

print("📄 Dockerfile:")
print(DOCKERFILE_EXAMPLE)
print("\n" + "="*60)
print("\n📄 requirements.txt:")
print(REQUIREMENTS_EXAMPLE)

---
## 📝 8. Summary & Quick Reference

### Model Summary

| Model | Task | Activation | Output |
|-------|------|------------|--------|
| ABSA | Aspect-Based Sentiment | Softmax | Per-aspect sentiment |
| Intent | Multi-label Classification | **Sigmoid** | Multiple labels |
| Sentiment | Single-label Classification | Softmax | Single label |

### Quick Usage

```python
# Initialize
nlp = ThaiNLPService()

# Full analysis
result = nlp.analyze_all("ข้อความ")

# Individual services
nlp.sentiment.analyze("ข้อความ")
nlp.intent.classify("ข้อความ", threshold=0.5)
nlp.absa.analyze("ข้อความ")

# Batch processing
nlp.sentiment.analyze(["text1", "text2", "text3"])
```

### Key Notes for Production

1. **Memory**: แต่ละ model ใช้ ~500MB-1GB RAM
2. **GPU**: ถ้ามี CUDA จะใช้อัตโนมัติ
3. **Singleton**: ใช้ singleton pattern เพื่อไม่ต้องโหลด model ซ้ำ
4. **Lazy Loading**: ใช้ `lazy_load=True` ถ้าไม่ต้องการโหลดทุก model ตอน startup

In [ ]:
print("\n" + "="*60)
print("✅ NOTEBOOK COMPLETE")
print("="*60)
print("\n📌 Services available:")
print("   • nlp.sentiment - Sentiment Analysis")
print("   • nlp.intent    - Intent Classification")
print("   • nlp.absa      - Aspect-Based Sentiment")
print("   • nlp.analyze_all() - Combined analysis")
print("\n🚀 Ready for FastAPI integration!")